# Train Per-Type Mask R-CNN Models — Fiber / Film / Fragment (Kaggle)

Trains **3 separate binary Mask R-CNN models**, one for each microplastic type,
using SAM-annotated crops.

## Kaggle Setup

1. **Upload 3 datasets** as Kaggle Datasets (each with `images/`, `masks/`, `annotations.json`):
   - `mp-crops-fiber-sam` — fiber crops
   - `mp-crops-film-sam` — film crops
   - `mp-crops-fragment-sam` — fragment crops
   - *or* upload all 3 in one dataset `mp-crops-per-type-sam` with subfolders

2. **Add all datasets** to notebook via sidebar → Add Data

3. **Enable GPU** (Settings → Accelerator → GPU T4 x2)

4. **Enable Internet** (Settings → Internet → On)

## Where Models Are Saved

```
/kaggle/working/
├── models/
│   ├── maskrcnn_fiber/maskrcnn_best.pth      ← DOWNLOAD
│   ├── maskrcnn_film/maskrcnn_best.pth       ← DOWNLOAD
│   └── maskrcnn_fragment/maskrcnn_best.pth   ← DOWNLOAD
└── training_loss.png
```

After training: **Save Version** → download from Output tab.

## 1. Install & Import

In [ ]:
!pip install -q albumentations opencv-python-headless

In [ ]:
import os, json, random, cv2, shutil, numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuration & Data Copy

In [ ]:
# ────────────────────────────────────────────────────────────────
# >>> CHANGE THESE to match your Kaggle Dataset names <<<
# ────────────────────────────────────────────────────────────────
# Option A: 3 separate datasets
DATASET_MAP = {
    'fiber':    'mp-crops-fiber-sam',
    'film':     'mp-crops-film-sam',
    'fragment': 'mp-crops-fragment-sam',
}
# Option B: single dataset with subfolders
SINGLE_DATASET = 'mp-crops-per-type-sam'  # set to None if using Option A
# ────────────────────────────────────────────────────────────────

TYPES = ['fiber', 'film', 'fragment']
LOCAL_ROOT = Path('/kaggle/working/mp_data')
MODEL_ROOT = Path('/kaggle/working/models')

# Copy data to writable working dir
for t in TYPES:
    dst = LOCAL_ROOT / f'crops_{t}_sam'
    if dst.exists():
        print(f'{t}: already copied'); continue

    # Try single dataset with subfolders first
    if SINGLE_DATASET:
        base = Path(f'/kaggle/input/{SINGLE_DATASET}')
        candidates = [
            base / f'crops_{t}_sam',
            base / t,
        ]
    else:
        candidates = []

    # Then try separate datasets
    ds = DATASET_MAP.get(t)
    if ds:
        sep = Path(f'/kaggle/input/{ds}')
        candidates += [sep, sep / f'crops_{t}_sam']

    src = None
    for c in candidates:
        if (c / 'annotations.json').exists():
            src = c; break
        found = list(c.rglob('annotations.json')) if c.exists() else []
        if found:
            src = found[0].parent; break

    assert src, f"Could not find data for '{t}'. Available: {os.listdir('/kaggle/input/')}"
    print(f'{t}: copying {src} → {dst}...')
    shutil.copytree(str(src), str(dst))
    print(f'  Done.')

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
print('\nAll data ready.')

## 3. Verify Data

In [ ]:
for t in TYPES:
    d = LOCAL_ROOT / f'crops_{t}_sam'
    imgs  = len(list((d/'images').glob('*.png')))  if (d/'images').exists()  else 0
    masks = len(list((d/'masks').glob('*.png')))   if (d/'masks').exists()   else 0
    ann   = (d / 'annotations.json').exists()
    ok = 'OK' if (imgs > 0 and masks > 0 and ann) else 'MISSING'
    print(f'  [{ok}] {t:>10}: {imgs} images, {masks} masks, annotations={ann}')

## 4. Dataset & Model Definitions

In [ ]:
NUM_CLASSES = 2
CROP_SIZE = 128
BATCH_SIZE = 8
LR = 0.001
EPOCHS = 50


class SingleTypeCropDataset(Dataset):
    def __init__(self, crops_dir, transforms=None, max_samples=None):
        self.crops_dir = Path(crops_dir)
        self.transforms = transforms
        self.images_dir = self.crops_dir / 'images'
        self.masks_dir  = self.crops_dir / 'masks'
        with open(self.crops_dir / 'annotations.json') as f:
            self.annotations = json.load(f)
        self.samples = [n for n in self.annotations if (self.images_dir / n).exists()]
        if max_samples and max_samples < len(self.samples):
            random.seed(42)
            self.samples = sorted(random.sample(self.samples, max_samples))
        print(f'  [{self.crops_dir.name}] {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        ann  = self.annotations[name]
        image = cv2.cvtColor(cv2.imread(str(self.images_dir / name)), cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        mask = None
        mf = ann.get('mask_file')
        if mf and (self.masks_dir / mf).exists():
            raw = cv2.imread(str(self.masks_dir / mf), cv2.IMREAD_GRAYSCALE)
            if raw is not None: mask = (raw > 127).astype(np.uint8)
        if mask is None:
            dm = self.masks_dir / name.replace('.png', '_mask.png')
            if dm.exists():
                raw = cv2.imread(str(dm), cv2.IMREAD_GRAYSCALE)
                if raw is not None: mask = (raw > 127).astype(np.uint8)
        if mask is None:
            mask = np.zeros((h, w), np.uint8)
            rb = ann.get('rel_box')
            cx, cy = ((rb[0]+rb[2])//2, (rb[1]+rb[3])//2) if rb else (w//2, h//2)
            ax, ay = ((rb[2]-rb[0])//2, (rb[3]-rb[1])//2) if rb else (int(w*0.4), int(h*0.4))
            if ax > 0 and ay > 0: cv2.ellipse(mask, (cx,cy), (ax,ay), 0, 0, 360, 1, -1)
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        ys, xs = np.where(mask > 0)
        box = [xs.min(), ys.min(), xs.max(), ys.max()] if len(xs) > 0 else [min(h,w)//10]*2 + [w-min(h,w)//10, h-min(h,w)//10]
        boxes  = np.array([box], dtype=np.float32)
        labels = np.array([1], dtype=np.int64)
        masks  = np.array([mask], dtype=np.uint8)

        if self.transforms:
            t = self.transforms(image=image, bboxes=boxes.tolist(),
                                masks=list(masks), class_labels=labels.tolist())
            image = t['image']
            if len(t['bboxes']) > 0:
                boxes  = np.array(t['bboxes'], np.float32)
                labels = np.array(t['class_labels'], np.int64)
                masks  = np.array(t['masks'], np.uint8)
        else:
            image = torch.from_numpy(image.transpose(2,0,1)).float() / 255.0

        return image, {
            'boxes':   torch.as_tensor(boxes, dtype=torch.float32),
            'labels':  torch.as_tensor(labels, dtype=torch.int64),
            'masks':   torch.as_tensor(masks, dtype=torch.uint8),
            'image_id': torch.tensor([idx]),
            'area':    torch.as_tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
            'iscrowd': torch.zeros(len(boxes), dtype=torch.int64),
        }


def get_transforms(train=True):
    if train:
        return A.Compose([
            A.Resize(CROP_SIZE, CROP_SIZE),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.GaussNoise(var_limit=(10., 50.), p=0.3),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
        ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))
    return A.Compose([
        A.Resize(CROP_SIZE, CROP_SIZE),
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
    ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))


def collate_fn(batch): return tuple(zip(*batch))

def get_model(num_classes=NUM_CLASSES):
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    inf = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(inf, num_classes)
    inf_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, num_classes)
    return model

print('Definitions loaded')

## 5. Training Function

In [ ]:
BACKUP_EVERY = 10
BACKUP_ROOT  = Path('/kaggle/working/backups')

def train_type(mp_type):
    crops_dir = str(LOCAL_ROOT / f'crops_{mp_type}_sam')
    save_dir  = MODEL_ROOT / f'maskrcnn_{mp_type}'
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"="*60}')
    print(f'TRAINING — {mp_type.upper()}')
    print(f'  crops : {crops_dir}')
    print(f'  save  : {save_dir}')
    print(f'  backup: every {BACKUP_EVERY} epochs → {BACKUP_ROOT}')
    print(f'{"="*60}')

    dataset = SingleTypeCropDataset(crops_dir, get_transforms(True))
    loader  = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=2,
                         collate_fn=collate_fn, pin_memory=True)

    model = get_model().to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optim  = torch.optim.AdamW(params, lr=LR, weight_decay=5e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=LR*0.01)

    best_loss = float('inf')
    history = []

    for epoch in range(EPOCHS):
        model.train(); epoch_loss = 0.0
        pbar = tqdm(loader, desc=f'[{mp_type}] Epoch {epoch+1}/{EPOCHS}', leave=False)
        for images, targets in pbar:
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            if not all(len(t['boxes']) > 0 for t in targets): continue
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())
            optim.zero_grad(); losses.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); optim.step()
            epoch_loss += losses.item(); pbar.set_postfix(loss=f'{losses.item():.4f}')

        sched.step()
        avg = epoch_loss / max(len(loader), 1)
        history.append(avg)
        ckpt = dict(epoch=epoch+1, mp_type=mp_type, num_classes=NUM_CLASSES,
                    model_state_dict=model.state_dict(),
                    optimizer_state_dict=optim.state_dict(), loss=avg, history=history)
        torch.save(ckpt, str(save_dir / 'maskrcnn_latest.pth'))
        tag = ''
        if avg < best_loss:
            best_loss = avg
            torch.save(ckpt, str(save_dir / 'maskrcnn_best.pth'))
            tag = ' * best'

        # Auto-backup
        if (epoch + 1) % BACKUP_EVERY == 0:
            backup_dir = BACKUP_ROOT / mp_type / f'epoch_{epoch+1:04d}'
            backup_dir.mkdir(parents=True, exist_ok=True)
            for bname in ('maskrcnn_best.pth', 'maskrcnn_latest.pth'):
                src = save_dir / bname
                if src.exists():
                    shutil.copy2(str(src), str(backup_dir / bname))
            print(f'  [AUTO-BACKUP] {mp_type} epoch {epoch+1} → {backup_dir}')

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:>3}/{EPOCHS}  loss={avg:.4f}{tag}')

    print(f'\n{mp_type} done — best loss {best_loss:.4f}')
    return history

print('Training function ready')


## 6. Train All 3 Models

In [ ]:
for t in TYPES:
    d = LOCAL_ROOT / f'crops_{t}_sam' / 'images'
    n = len(list(d.glob('*.png'))) if d.exists() else 0
    print(f'  {t}: {n} samples')

all_history = {}
for mp_type in TYPES:
    all_history[mp_type] = train_type(mp_type)

print(f'\n{"="*60}')
print('ALL 3 MODELS TRAINED')
for t in TYPES:
    print(f'  {t:>10}: {MODEL_ROOT}/maskrcnn_{t}/maskrcnn_best.pth')
print(f'{"="*60}')

## 7. Training Loss Curves

In [ ]:
# ==============================================================================
# 7. TRAINING LOSS CURVES (self-contained)
# ==============================================================================

import torch, matplotlib.pyplot as plt
from pathlib import Path

TYPES      = ['fiber', 'film', 'fragment']
MODEL_ROOT = Path('/kaggle/working/models')
BACKUP_ROOT = Path('/kaggle/working/backups')

all_history = {}
for t in TYPES:
    ckpt_path = MODEL_ROOT / f'maskrcnn_{t}' / 'maskrcnn_best.pth'
    if not ckpt_path.exists():
        # Try backup
        backups = sorted(BACKUP_ROOT.glob(f'{t}/epoch_*/maskrcnn_best.pth')) if BACKUP_ROOT.exists() else []
        ckpt_path = backups[-1] if backups else ckpt_path
    if ckpt_path.exists():
        ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
        all_history[t] = ckpt.get('history', [])
        print(f'  {t}: loaded history ({len(all_history[t])} epochs)')
    else:
        print(f'  {t}: checkpoint not found, skipping')

fig, ax = plt.subplots(figsize=(10, 5))
for t in TYPES:
    if t in all_history and all_history[t]:
        ax.plot(range(1, len(all_history[t])+1), all_history[t], label=t)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Per-Type Mask R-CNN Training Loss'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_loss.png', dpi=150)
plt.show()


## 8. Sanity Check — Predictions

In [ ]:
# ==============================================================================
# 8. SANITY CHECK — PREDICTIONS (self-contained)
# ==============================================================================

import random, torch, cv2, numpy as np, matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from pathlib import Path
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

NUM_CLASSES = 2
CROP_SIZE   = 128
TYPES       = ['fiber', 'film', 'fragment']
MODEL_ROOT  = Path('/kaggle/working/models')
LOCAL_ROOT  = Path('/kaggle/working/mp_data')
BACKUP_ROOT = Path('/kaggle/working/backups')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_model():
    m = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES)
    inf_m = m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, NUM_CLASSES)
    return m

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row, mp_type in enumerate(TYPES):
    ckpt_path = MODEL_ROOT / f'maskrcnn_{mp_type}' / 'maskrcnn_best.pth'
    if not ckpt_path.exists():
        backups = sorted(BACKUP_ROOT.glob(f'{mp_type}/epoch_*/maskrcnn_best.pth')) if BACKUP_ROOT.exists() else []
        if backups:
            ckpt_path = backups[-1]
        else:
            print(f'[SKIP] {mp_type}: no checkpoint found')
            for col in range(4):
                axes[row][col].text(0.5,0.5,'No model',ha='center',va='center'); axes[row][col].axis('off')
            continue

    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model = get_model().to(device)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()

    imgs_dir = LOCAL_ROOT / f'crops_{mp_type}_sam' / 'images'
    imgs = sorted(imgs_dir.glob('*.png'))
    samples = random.sample(imgs, min(4, len(imgs)))

    for col, img_path in enumerate(samples):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img, (CROP_SIZE, CROP_SIZE))
        tensor = F.to_tensor(resized).unsqueeze(0).to(device)
        with torch.no_grad(): out = model(tensor)[0]
        ax = axes[row][col]; ax.imshow(resized)
        if len(out['masks']) > 0:
            best = out['scores'].argmax()
            mask = out['masks'][best, 0].cpu().numpy() > 0.5
            score = out['scores'][best].item()
            ax.contour(mask, colors='lime', linewidths=1)
            ax.set_title(f'{mp_type} ({score:.2f})', fontsize=10)
        else: ax.set_title(f'{mp_type} (no det)', fontsize=10)
        ax.axis('off')
plt.suptitle('Per-Type Mask R-CNN Predictions', fontsize=14)
plt.tight_layout(); plt.show()


## 9. Download

Models are in `/kaggle/working/models/`:

| Model | Path |
|-------|------|
| Fiber | `models/maskrcnn_fiber/maskrcnn_best.pth` |
| Film | `models/maskrcnn_film/maskrcnn_best.pth` |
| Fragment | `models/maskrcnn_fragment/maskrcnn_best.pth` |

Click **Save Version** then download from the **Output** tab.